In [1]:
!pip install fastapi \
uvicorn[standard] \
pandas \
faiss-cpu \
sentence-transformers \
"pyarrow>=16" \
"transformers>=4.44" \
accelerate \
bitsandbytes \
pyngrok \
nest_asyncio \
uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 45.2 MB/s eta 0:00:00:00:0100:01


In [3]:
"""
CSE488 Big Data Analytics — Term Project
A RAG-Based Expert System for Mobile/Laptop Recommendation
Backend (FastAPI + FAISS + Sentence-Transformers + LLM)

============================================================================
WHAT WAS FIXED / ADDED vs. your original main.py
============================================================================
Bugs (would have crashed the app):
  1. `ngrok.disconnect(tunnel.public_url)` was called BEFORE `tunnel` was
     ever assigned -> guaranteed NameError on startup. Removed; a tunnel
     is only opened once, at the end, and only in Kaggle mode.
  2. `verify_grounding(...)` was called inside recommend_verified() but was
     never defined anywhere -> NameError on the first real request.
     Implemented below (see "GROUNDING / HALLUCINATION CHECK").
  3. `get_paired_specs(...)` was called inside recommend_full() but was
     never defined -> NameError on the first request with use_cache logic
     reached. Implemented below (see "COMMONLY PAIRED SPECS").
  4. `kaggle_secrets` / `pyngrok` / `nest_asyncio` were imported
     unconditionally, so the file could only ever run inside a Kaggle
     notebook. Now gated behind a RUNTIME env var so it also runs as a
     normal FastAPI service (uvicorn) anywhere else.
  5. DATA_DIR / OUT_DIR were hardcoded to /kaggle/... paths. Now read from
     environment variables with the old Kaggle paths kept as defaults, so
     nothing breaks for you on Kaggle but it's portable elsewhere too.
  6. On import, the code built ALL 25 (5 index types x 5 presets) FAISS
     indexes eagerly before the server even started — very slow / memory
     heavy, and pointless because `retrieve()` already lazily builds and
     caches whichever (index_type, preset) pair is actually requested.
     Removed the eager loop; only a single default index is warmed up on
     startup, everything else builds on first use exactly as your
     retrieve() function already intended.

Missing mandatory deliverables (per the project description), now added:
  7. Section 3.3 asks you to "compare at least two FAISS index types on
     recall@k and query latency" — there was no code that actually does
     this comparison. Added `benchmark_indexes()` + `POST /benchmark`.
  8. Section 3.6 asks for a hand-labeled query set (>=30 queries) with
     precision@k / recall@k reporting — there was no evaluation harness
     at all. Added `evaluate_retrieval()` + `POST /evaluate`, which reads
     a JSON file of {"query": ..., "relevant_titles": [...]} you fill in
     with your team's real labels.
  9. Grounding check (3.4: "unsupported/hallucinated answers are not
     acceptable") referenced but not implemented — implemented as a cheap
     deterministic post-hoc check (price + device-name grounding).
 10. `get_paired_specs` (stretch goal in section 4, FP-Growth-style
     "commonly paired specifications") — implemented as the *serving*
     side of that feature: it prefers a `paired_specs.parquet` produced by
     an offline Spark `pyspark.ml.fpm.FPGrowth` job (which is where the
     actual FP-Growth computation belongs per section 3.2/4), and falls
     back to an on-the-fly co-occurrence count if that file isn't present
     yet, so the API never crashes even before you've run the Spark job.
 11. Section 4 also asks for the semantic cache to be "benchmarked for
     latency and hit-rate improvement" — the cache itself (SemanticCache)
     existed, but nothing measured it. Added `benchmark_semantic_cache()`
     + `POST /benchmark_cache`, which runs a synthetic query stream
     (originals + paraphrased near-duplicates) through a fresh cache
     instance and reports hit rate, avg latency for hits vs. misses, and
     the resulting speedup factor.

Section 4 stretch goals — where each one lives in this file:
  - Hybrid search (metadata filter + vector similarity): `extract_filters()`
    + `retrieve()`.
  - FP-Growth spec co-occurrence / "commonly paired specs": `get_paired_specs()`
    + `GET /paired_specs/{cpu}`.
  - Semantic caching, benchmarked: `SemanticCache` + `recommend_full()`
    + `benchmark_semantic_cache()` + `POST /benchmark_cache`.

Follow-up fixes after a second pass against the project PDF:
 12. Markdown was only "requested" via one prompt sentence, with nothing
     enforcing it — the model would sometimes wrap output in a
     ```markdown code fence (which renders as a literal code block, not
     formatted text). `generate_recommendation()` now gives explicit
     formatting rules and `_strip_markdown_fence()` strips any outer fence
     the model still adds.
 13. No device ever carried a source link, and the LLM had no way to cite
     one even if the dataset had it. Added auto-detection of a source-link
     column (`source_url`/`source_link`/`product_url`/`url`/`link` — first
     match wins) in `retrieve()`, exposed uniformly as `source_url` on
     every device in `retrieved_context`. The LLM prompt now includes each
     device's link when present and is told to cite it as a Markdown link
     `[title](source_url)` — and `verify_grounding()` now also flags any
     link the model mentions that doesn't match a retrieved device's real
     source_url, catching invented URLs the same way it already caught
     invented prices. If your dataset has no such column yet, source_url
     is simply null everywhere — nothing crashes, but add one of those
     column names to your collected data to get links end-to-end.
 14. `retrieved_context` could contain raw NaN for any optional/missing
     field (most likely `source_url`), which `to_dict(orient="records")`
     leaves as a Python float NaN — invalid JSON. Added `_records_safe()`
     and used it everywhere `retrieved_context` is serialized.

Everything else (ANNIndex subclasses, extract_filters, SemanticCache,
recommend_* functions) is your original design/logic, kept as-is except
for the bug fixes and safety guards noted inline.
============================================================================
"""

import json
import os
import re
import threading
import time
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
from typing import Optional

import faiss
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration (env-overridable so this runs on Kaggle AND anywhere else)
# ---------------------------------------------------------------------------
RUNTIME = os.environ.get("RUNTIME", "kaggle").lower()  # "kaggle" or "local"

DATA_DIR = os.environ.get(
    "DATA_DIR", "/kaggle/input/datasets/mrnotalent/laptop-embedding/"
)
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/")

PARQUET_PATH = os.environ.get(
    "PARQUET_PATH",
    os.path.join(DATA_DIR, "laptop_chunks_embeddings_with_lineage.parquet"),
)
# Optional: output of an offline Spark FPGrowth job (section 3.2/4). If this
# file doesn't exist yet, get_paired_specs() falls back to an on-the-fly
# groupby over the loaded device table so the API still works.
PAIRED_SPECS_PATH = os.environ.get(
    "PAIRED_SPECS_PATH", os.path.join(DATA_DIR, "paired_specs.parquet")
)

# Hand-labeled evaluation set for section 3.6 (precision@k / recall@k).
# JSON list of: {"query": "...", "relevant_titles": ["Title A", "Title B"]}
# You need >= 30 hand-labeled queries in here before grading.
EVAL_QUERY_SET_PATH = os.environ.get(
    "EVAL_QUERY_SET_PATH", os.path.join(OUT_DIR, "eval_query_set.json")
)

REQUIRED_COLUMNS = [
    "row_uid",
    "title",
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "display",
    "battery",
    "chunk_text",
    "lineage",
    "embedding",
]

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(
        f"Could not find embeddings parquet at {PARQUET_PATH}. "
        f"Set the PARQUET_PATH env var, or DATA_DIR if you kept the default filename."
    )

df = pd.read_parquet(PARQUET_PATH)

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Parquet at {PARQUET_PATH} is missing required columns: {missing_cols}. "
        f"Re-check your Spark pipeline output schema (section 3.2)."
    )

embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
device_level = df.drop_duplicates(subset="row_uid").copy()

print(
    f"[startup] loaded {df.shape[0]} chunk rows, {device_level.shape[0]} unique devices, "
    f"embedding dim={embeddings.shape[1]}"
)

# ---------------------------------------------------------------------------
# Source link per device
# ---------------------------------------------------------------------------
# Your dataset schema (section 3.1) doesn't mandate a specific column name
# for a product/source URL, so this auto-detects whichever common name your
# collected data used. If your collection scripts (scraping / manual
# compilation) captured the page a listing came from, name that column one
# of these (in priority order) and it will automatically flow through every
# API response as "source_url". If none of these columns exist, source_url
# is simply omitted/null for every device — nothing breaks, but you won't
# get links until your dataset actually has one of these columns.
SOURCE_LINK_CANDIDATES = ["source_url", "source_link", "product_url", "url", "link"]


def _detect_source_link_column(dataframe: pd.DataFrame):
    for col in SOURCE_LINK_CANDIDATES:
        if col in dataframe.columns:
            return col
    return None


SOURCE_LINK_COLUMN = _detect_source_link_column(df)
if SOURCE_LINK_COLUMN:
    print(
        f"[startup] detected source-link column '{SOURCE_LINK_COLUMN}' — "
        f"will be exposed as 'source_url' on every retrieved device"
    )
else:
    print(
        f"[startup] no source-link column found (looked for: {SOURCE_LINK_CANDIDATES}) — "
        f"'source_url' will be null for every device. Add one of these column names "
        f"to your collected dataset to enable device source links."
    )


def _records_safe(dataframe: pd.DataFrame):
    """to_dict(orient='records'), but NaN/NaT -> None so the response is valid JSON
    (pandas leaves NaN as a float NaN, which json can only emit as invalid 'NaN' literal).
    """
    return dataframe.where(pd.notna(dataframe), None).to_dict(orient="records")


try:
    paired_specs_df = pd.read_parquet(PAIRED_SPECS_PATH)
    print(
        f"[startup] loaded FP-Growth paired-specs table from {PAIRED_SPECS_PATH} "
        f"({len(paired_specs_df)} rules)"
    )
except Exception:
    paired_specs_df = None
    print(
        f"[startup] no paired-specs table found at {PAIRED_SPECS_PATH} — "
        f"'commonly paired specs' will fall back to on-the-fly co-occurrence counts. "
        f"Run the offline Spark FPGrowth job to populate this file for the real stretch-goal credit."
    )

# ---------------------------------------------------------------------------
# LLM (quantized generator)
# ---------------------------------------------------------------------------
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

MODEL_ID = os.environ.get("LLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    return_full_text=False,
)

print(f"[startup] LLM device map: {model.hf_device_map}")


# ---------------------------------------------------------------------------
# 3.4 Hybrid search — metadata filter extraction (stretch goal + retrieval)
# ---------------------------------------------------------------------------
def extract_filters(query: str):
    q = query.lower()
    mask = pd.Series(True, index=device_level.index)
    applied = []

    m = (
        re.search(r"under\s*\$?(\d+)", q)
        or re.search(r"less than\s*\$?(\d+)", q)
        or re.search(r"budget.*?\$?(\d+)", q)
    )
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] < price
        applied.append(f"price_usd < {price}")

    m = re.search(r"over\s*\$?(\d+)", q) or re.search(r"above\s*\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] > price
        applied.append(f"price_usd > {price}")

    m = re.search(r"(\d+)\s*gb\s*ram", q) or re.search(r"at least\s*(\d+)\s*gb", q)
    if m:
        ram = float(m.group(1))
        mask &= device_level["ram_gb"] >= ram
        applied.append(f"ram_gb >= {ram}")

    if any(w in q for w in ["nvidia", "rtx", "geforce", "gaming"]):
        mask &= device_level["gpu"].str.contains(
            "NVIDIA|RTX|GeForce", case=False, na=False
        )
        applied.append("gpu contains NVIDIA/RTX/GeForce")

    if "amd" in q or "ryzen" in q:
        mask &= device_level["cpu"].str.contains("Ryzen|AMD", case=False, na=False)
        applied.append("cpu contains Ryzen/AMD")

    for cpu_kw in ["i9", "i7", "i5", "i3"]:
        if cpu_kw in q:
            mask &= device_level["cpu"].str.contains(cpu_kw, case=False, na=False)
            applied.append(f"cpu contains {cpu_kw}")

    if "ssd" in q or "nvme" in q:
        mask &= device_level["storage"].str.contains(
            "SSD|PCIe|NVMe", case=False, na=False
        )
        applied.append("storage is SSD/PCIe/NVMe")

    return mask, applied


# ---------------------------------------------------------------------------
# 3.3 FAISS index types
# ---------------------------------------------------------------------------
class ANNIndex(ABC):
    @abstractmethod
    def build(self, embeddings: np.ndarray):
        pass

    @abstractmethod
    def search(self, query_vec: np.ndarray, k: int):
        pass


class FlatL2(ANNIndex):
    """Exact baseline — used both as a retrieval option and as the recall@k
    ground truth for benchmark_indexes()."""

    def __init__(self):
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)


class IVFFlat(ANNIndex):
    def __init__(self, nlist=100, nprobe=10):
        self.nlist = nlist
        self.nprobe = nprobe
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]
        nlist = max(1, min(self.nlist, len(embeddings)))

        quantizer = faiss.IndexFlatL2(dim)
        self.index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
        self.index.train(embeddings)
        self.index.add(embeddings)
        self.index.nprobe = min(self.nprobe, nlist)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)


class PQ(ANNIndex):
    def __init__(self, m=8, bits=8):
        self.m = m
        self.bits = bits
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]
        if dim % self.m != 0:
            raise ValueError(
                f"Embedding dimension ({dim}) must be divisible by m ({self.m})"
            )

        # FAISS PQ training wants at least ~ (2**bits) points per sub-quantizer
        # to fit clusters reliably. This is not a hard requirement but avoids
        # silent, badly-fit quantizers on tiny datasets.
        min_recommended = 2**self.bits
        if len(embeddings) < min_recommended:
            print(
                f"[warn] PQ(m={self.m}, bits={self.bits}) is being trained on only "
                f"{len(embeddings)} vectors; recommended minimum is {min_recommended}. "
                f"Quantization quality may be poor."
            )

        self.index = faiss.IndexPQ(dim, self.m, self.bits)
        self.index.train(embeddings)
        self.index.add(embeddings)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)


def _safe_nlist(requested_nlist: int, n_vectors: int) -> int:
    max_nlist = max(1, n_vectors // 39)
    return min(requested_nlist, max_nlist)


class IVFPQ(ANNIndex):
    def __init__(self, nlist=100, nprobe=10, m=8, bits=8):
        self.nlist = nlist
        self.nprobe = nprobe
        self.m = m
        self.bits = bits
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]
        if dim % self.m != 0:
            raise ValueError(
                f"Embedding dimension ({dim}) must be divisible by m ({self.m})"
            )

        nlist = max(1, _safe_nlist(self.nlist, len(embeddings)))

        quantizer = faiss.IndexFlatL2(dim)
        self.index = faiss.IndexIVFPQ(quantizer, dim, nlist, self.m, self.bits)
        self.index.train(embeddings)
        self.index.add(embeddings)
        self.index.nprobe = min(self.nprobe, nlist)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)


class HNSW(ANNIndex):
    def __init__(self, M=32, ef_construction=40, ef_search=16):
        self.M = M
        self.ef_construction = ef_construction
        self.ef_search = ef_search
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]
        self.index = faiss.IndexHNSWFlat(dim, self.M, faiss.METRIC_L2)
        self.index.hnsw.efConstruction = self.ef_construction
        self.index.hnsw.efSearch = self.ef_search
        self.index.add(embeddings)  # HNSW needs no training

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)


class IndexPreset(str, Enum):
    MAX = "max"
    EXTRA = "extra"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


def create_index(preset, index_type: str = "ivfpq") -> ANNIndex:
    if isinstance(preset, str):
        preset = IndexPreset(preset.lower())
    index_type = index_type.lower()

    # accuracy-leaning -> speed/memory-leaning as preset goes MAX -> LOW
    params = {
        IndexPreset.MAX: dict(ivf=(32, 32), pq=(16, 4), hnsw=(32, 200, 128)),
        IndexPreset.EXTRA: dict(ivf=(24, 16), pq=(16, 4), hnsw=(24, 150, 64)),
        IndexPreset.HIGH: dict(ivf=(16, 8), pq=(8, 4), hnsw=(16, 100, 32)),
        IndexPreset.MEDIUM: dict(ivf=(8, 4), pq=(8, 4), hnsw=(12, 80, 16)),
        IndexPreset.LOW: dict(ivf=(4, 2), pq=(4, 4), hnsw=(8, 40, 8)),
    }[preset]

    if index_type == "flat":
        return FlatL2()
    if index_type == "ivfflat":
        nlist, nprobe = params["ivf"]
        return IVFFlat(nlist=nlist, nprobe=nprobe)
    if index_type == "pq":
        m, bits = params["pq"]
        return PQ(m=m, bits=bits)
    if index_type == "ivfpq":
        nlist, nprobe = params["ivf"]
        m, bits = params["pq"]
        return IVFPQ(nlist=nlist, nprobe=nprobe, m=m, bits=bits)
    if index_type == "hnsw":
        M, ef_c, ef_s = params["hnsw"]
        return HNSW(M=M, ef_construction=ef_c, ef_search=ef_s)

    raise ValueError(
        f"Unsupported preset/index combination: {preset.value}/{index_type}"
    )


# Cache of built indexes, keyed by [index_type][preset]. Populated lazily by
# retrieve() the first time a given combination is actually requested — NOT
# eagerly for all 25 combinations at import time (that was the old behavior
# and made startup take forever for no benefit).
indexes = {"ivfpq": {}, "ivfflat": {}, "pq": {}, "hnsw": {}, "flat": {}}


def retrieve(
    query: str,
    k: int = 5,
    index_type: str = "ivfpq",
    preset: str = "medium",
    candidate_k: int = 100,
):
    index_type = index_type.lower()
    preset = preset.lower()

    # 1. Metadata filters (hybrid search, section 4 stretch goal)
    mask, applied_filters = extract_filters(query)
    candidate_uids = set(device_level.loc[mask, "row_uid"])

    # 2. Fallback if filters are too strict
    if len(candidate_uids) < k:
        candidate_uids = set(device_level["row_uid"])
        applied_filters = applied_filters + ["(filters relaxed — too few matches)"]

    # 3. Get cached / lazily build index
    if index_type not in indexes:
        raise ValueError(
            f"Unknown index type: {index_type}. Available: {list(indexes.keys())}"
        )

    if preset in indexes[index_type]:
        index = indexes[index_type][preset]
    else:
        print(f"[index] building {index_type}/{preset} (first use)...")
        t0 = time.perf_counter()
        index = create_index(preset=preset, index_type=index_type)
        index.build(embeddings)
        indexes[index_type][preset] = index
        print(f"[index] built {index_type}/{preset} in {time.perf_counter() - t0:.2f}s")

    # 4. Encode query
    query_vec = embed_model.encode([query]).astype("float32")

    # 5. Over-fetch candidates so metadata filtering has room to work
    search_k = min(candidate_k, len(df))
    _, global_idx = index.search(query_vec, search_k)
    global_idx = global_idx[0]
    global_idx = global_idx[global_idx >= 0]  # drop invalid FAISS hits (-1)

    # 6. Apply metadata filter to vector results
    filtered_idx = []
    for idx in global_idx:
        row_uid = df.iloc[idx]["row_uid"]
        if row_uid in candidate_uids:
            filtered_idx.append(idx)
        if len(filtered_idx) >= k:
            break

    # 7. Relax if still not enough
    if len(filtered_idx) < k:
        applied_filters = applied_filters + [
            "(filters relaxed — insufficient vector matches)"
        ]
        filtered_idx = global_idx[:k]

    # 8. Build result dataframe (+ normalized source_url, section 3.4)
    select_cols = [
        "title",
        "price_usd",
        "cpu",
        "ram_gb",
        "storage",
        "gpu",
        "display",
        "battery",
        "chunk_text",
        "lineage",
    ]
    if SOURCE_LINK_COLUMN:
        select_cols.append(SOURCE_LINK_COLUMN)

    results = df.iloc[filtered_idx][select_cols].copy()

    if SOURCE_LINK_COLUMN and SOURCE_LINK_COLUMN != "source_url":
        results = results.rename(columns={SOURCE_LINK_COLUMN: "source_url"})
    if "source_url" not in results.columns:
        results["source_url"] = None

    return results, applied_filters


# ---------------------------------------------------------------------------
# 3.3 Recall@k / latency comparison across FAISS index types (was missing)
# ---------------------------------------------------------------------------
def benchmark_indexes(
    sample_size: int = 200,
    k: int = 10,
    index_types=None,
    preset: str = "medium",
    seed: int = 42,
):
    """
    Satisfies section 3.3: 'Implement and compare at least two FAISS index
    types ... on recall@k and query latency.'

    Samples `sample_size` random vectors from the dataset itself as queries,
    uses exact FlatL2 search as ground truth, and measures each other index
    type's overlap with that ground truth (recall@k) plus wall-clock query
    latency, all at the same preset (so the comparison is apples-to-apples).
    """
    if index_types is None:
        index_types = ["flat", "ivfflat", "pq", "ivfpq", "hnsw"]

    rng = np.random.default_rng(seed)
    n = len(embeddings)
    sample_size = min(sample_size, n)
    query_idx = rng.choice(n, size=sample_size, replace=False)
    query_vecs = embeddings[query_idx]

    def get_or_build(itype):
        if preset in indexes[itype]:
            return indexes[itype][preset]
        idx = create_index(preset=preset, index_type=itype)
        idx.build(embeddings)
        indexes[itype][preset] = idx
        return idx

    gt_index = get_or_build("flat")
    _, gt_neighbors = gt_index.search(query_vecs, k)

    report = []
    for itype in index_types:
        idx = get_or_build(itype)

        t0 = time.perf_counter()
        _, result_neighbors = idx.search(query_vecs, k)
        elapsed = time.perf_counter() - t0

        recalls = []
        for gt_row, res_row in zip(gt_neighbors, result_neighbors):
            gt_set = set(gt_row[gt_row >= 0].tolist())
            res_set = set(res_row[res_row >= 0].tolist())
            if gt_set:
                recalls.append(len(gt_set & res_set) / len(gt_set))
        avg_recall = float(np.mean(recalls)) if recalls else 0.0

        report.append(
            {
                "index_type": itype,
                "preset": preset,
                "avg_recall_at_k": round(avg_recall, 4),
                "total_query_time_sec": round(elapsed, 4),
                "avg_latency_ms_per_query": round((elapsed / sample_size) * 1000, 4),
            }
        )

    return sorted(report, key=lambda r: -r["avg_recall_at_k"])


# ---------------------------------------------------------------------------
# 3.4 Generation
# ---------------------------------------------------------------------------
_MD_FENCE_RE = re.compile(r"^```(?:markdown|md)?\s*\n(.*)\n```$", re.DOTALL)


def _strip_markdown_fence(text: str) -> str:
    """Instruction-tuned models frequently wrap 'respond in markdown' output in a
    ```markdown ... ``` code fence, which then renders as a literal code block
    instead of formatted markdown on the frontend. Strip that outer fence if present."""
    text = text.strip()
    m = _MD_FENCE_RE.match(text)
    return m.group(1).strip() if m else text


def generate_recommendation(query: str, retrieved_df: pd.DataFrame) -> str:
    context_blocks = []
    for _, row in retrieved_df.iterrows():
        line = (
            f"- {row['title']} | ${row['price_usd']:.2f} | {row['cpu']} | {row['ram_gb']}GB RAM | "
            f"{row['storage']} | {row['gpu']} | {row['display']}"
        )
        source_url = row.get("source_url")
        if isinstance(source_url, str) and source_url.strip():
            line += f" | source: {source_url}"
        context_blocks.append(line)
    context = "\n".join(context_blocks)

    user_prompt = f"""
        You are a laptop recommendation assistant. Base your answer ONLY on the devices listed
        below - do not invent specs, devices, or source links not present here.

        User request: {query}

        Retrieved candidate devices:
        {context}

        Formatting rules:
        - Respond in clean Markdown only: use headers, bold key specs, and a table or bullet
          list for the devices — never a single paragraph of free text.
        - Do NOT wrap your response in a ```markdown or ``` code fence. Output raw Markdown directly.
        - For each device that has a "source:" link listed above, reference that device as a
          Markdown link, e.g. [Device Title](source_url). If a device has no source link above,
          just write its name as plain text — never invent a URL.

        Give a recommendation and justify the choice using only the specs shown above.
        Your answer will be forwarded directly to the user, so speak to them, not to me.
    """

    messages = [{"role": "user", "content": user_prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = llm_pipe(prompt_text)
    return _strip_markdown_fence(output[0]["generated_text"])


def recommend(query: str, k: int, index_type: str, preset: str):
    retrieved, applied_filters = retrieve(query, k, index_type, preset)
    answer = generate_recommendation(query, retrieved)
    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": _records_safe(retrieved.drop(columns=["lineage"])),
        "lineage": retrieved["lineage"].tolist(),
        "recommendation": answer,
    }


def parse_filter_condition(filter_str: str):
    """Turn an applied-filter string like 'price_usd < 600' into (column, operator, threshold).
    Returns None for non-numeric filters (e.g. 'gpu contains NVIDIA/RTX/GeForce')."""
    m = re.match(r"(\w+)\s*(>=|<=|>|<|==)\s*([\d.]+)", filter_str.strip())
    if not m:
        return None
    col, op, threshold = m.group(1), m.group(2), float(m.group(3))
    return (col, op, threshold)


def check_condition(actual, op, threshold):
    if op == "<":
        return actual < threshold
    if op == "<=":
        return actual <= threshold
    if op == ">":
        return actual > threshold
    if op == ">=":
        return actual >= threshold
    if op == "==":
        return actual == threshold
    return True


# ---------------------------------------------------------------------------
# 3.4 Grounding / hallucination check (was referenced, never implemented)
# ---------------------------------------------------------------------------
def verify_grounding(
    query: str, retrieved_df: pd.DataFrame, applied_filters, answer_text: str
):
    """
    Section 3.4: 'unsupported/hallucinated answers are not acceptable.'

    This is a cheap, deterministic post-hoc check (no extra LLM call
    needed) rather than a full fact-checking pass:
      1. Every dollar amount mentioned in the answer must be within ~2%
         (or $1, whichever is larger) of at least one retrieved device's
         price — catches invented prices.
      2. The answer should reference at least one retrieved device by
         name — catches answers that talk about devices not in context.
      3. Every Markdown link target ([text](url)) in the answer must match
         a retrieved device's source_url — catches invented source links.
    Returns (verified_facts: list[str], warnings: list[str]).
    """
    verified_facts = []
    warnings = []

    if retrieved_df.empty:
        warnings.append("No candidate devices were retrieved to ground the answer.")
        return verified_facts, warnings

    retrieved_titles = retrieved_df["title"].astype(str).tolist()
    retrieved_prices = retrieved_df["price_usd"].astype(float).tolist()
    retrieved_urls = [
        u
        for u in retrieved_df.get("source_url", pd.Series(dtype=object)).tolist()
        if isinstance(u, str) and u.strip()
    ]

    mentioned_prices = re.findall(r"\$\s?([\d,]+(?:\.\d+)?)", answer_text)
    for raw in mentioned_prices:
        val = float(raw.replace(",", ""))
        if any(abs(val - p) <= max(1.0, 0.02 * p) for p in retrieved_prices):
            verified_facts.append(
                f"Mentioned price ${val:g} matches a retrieved device."
            )
        else:
            warnings.append(
                f"Mentioned price ${val:g} does not match any retrieved device's price."
            )

    mentioned_urls = re.findall(r"\]\((https?://[^\s)]+)\)", answer_text)
    for url in mentioned_urls:
        if url in retrieved_urls:
            verified_facts.append(
                f"Linked URL {url} matches a retrieved device's source link."
            )
        else:
            warnings.append(
                f"Linked URL {url} does not match any retrieved device's source link."
            )

    answer_lower = answer_text.lower()
    referenced = [t for t in retrieved_titles if t and t.lower()[:15] in answer_lower]
    if referenced:
        verified_facts.append(
            f"Answer references retrieved device(s): {', '.join(referenced[:3])}."
        )
    else:
        warnings.append(
            "Answer does not clearly reference any retrieved device by name."
        )

    return verified_facts, warnings


def recommend_verified(query: str, index_type, preset, k: int = 5):
    retrieved, applied_filters = retrieve(query, k, index_type, preset)
    answer = generate_recommendation(query, retrieved)
    verified_facts, warnings = verify_grounding(
        query, retrieved, applied_filters, answer
    )

    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": _records_safe(retrieved.drop(columns=["lineage"])),
        "lineage": retrieved["lineage"].tolist(),
        "recommendation": answer,
        "grounding_check": {
            "verified_facts": verified_facts,
            "warnings": warnings,
            "passed": len(warnings) == 0,
        },
    }


# ---------------------------------------------------------------------------
# Section 4 stretch goal: commonly paired specifications (FP-Growth serving)
# ---------------------------------------------------------------------------
def get_paired_specs(cpu_value: str, top_n: int = 5):
    """
    Section 4 stretch goal: 'FP-Growth analysis on spec co-occurrence to
    power a "commonly paired specifications" insight feature.'

    The actual FP-Growth computation (pyspark.ml.fpm.FPGrowth over
    itemsets like {cpu, gpu, ram_gb, storage}) belongs in the offline Spark
    pipeline (section 3.2), writing its frequent itemsets / rules out to
    `paired_specs.parquet`. This function is the *serving* side:
      - If that parquet exists (loaded at startup as `paired_specs_df`),
        look up rules whose antecedent matches `cpu_value`.
      - Otherwise, fall back to an on-the-fly co-occurrence count over the
        already-loaded device table, so the API still works before you've
        run the Spark job.
    """
    if paired_specs_df is not None and "antecedent" in paired_specs_df.columns:
        rules = paired_specs_df[paired_specs_df["antecedent"] == cpu_value]
        if not rules.empty:
            rules = rules.sort_values("confidence", ascending=False).head(top_n)
            return rules.to_dict(orient="records")

    subset = device_level[device_level["cpu"] == cpu_value]
    if subset.empty:
        return []

    combo_counts = (
        subset.groupby(["gpu", "ram_gb", "storage"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(top_n)
    )

    total = len(subset)
    pairs = []
    for _, row in combo_counts.iterrows():
        pairs.append(
            {
                "gpu": row["gpu"],
                "ram_gb": row["ram_gb"],
                "storage": row["storage"],
                "support": round(row["count"] / total, 3),
                "count": int(row["count"]),
            }
        )
    return pairs


# ---------------------------------------------------------------------------
# Section 4 stretch goal: semantic cache
# ---------------------------------------------------------------------------
class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.92, max_size: int = 200):
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.embeddings = []
        self.entries = []

    def _normalize(self, vec):
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def lookup(self, query_embedding: np.ndarray):
        if not self.embeddings:
            return None, None
        q = self._normalize(query_embedding)
        sims = np.array([np.dot(q, e) for e in self.embeddings])
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])
        if best_sim >= self.threshold:
            matched_query, response = self.entries[best_idx]
            return response, {"matched_query": matched_query, "similarity": best_sim}
        return None, None

    def store(self, query: str, query_embedding: np.ndarray, response: dict):
        if len(self.embeddings) >= self.max_size:
            self.embeddings.pop(0)
            self.entries.pop(0)
        self.embeddings.append(self._normalize(query_embedding))
        self.entries.append((query, response))

    def __len__(self):
        return len(self.entries)


cache = SemanticCache(similarity_threshold=0.92)


def benchmark_semantic_cache(
    queries: Optional[list] = None,
    repeat_ratio: float = 0.5,
    index_type: str = "ivfpq",
    preset: str = "medium",
    k: int = 5,
    seed: int = 42,
):
    """
    Section 4 stretch goal: 'A semantic caching layer for similar past
    queries, benchmarked for latency and hit-rate improvement.'

    A SemanticCache existed before, but nothing measured whether it
    actually helps — this does that measurement.

    Builds a synthetic query stream (each query appears once as a
    guaranteed miss, then — with probability `repeat_ratio` — again as a
    lightly paraphrased near-duplicate, e.g. "gaming laptop under $1500"
    -> "looking for a gaming laptop under $1500", to exercise the
    *semantic* similarity match rather than exact-string caching) and runs
    it through a fresh cache instance, timing every lookup.

    If no `queries` are passed, pulls from your hand-labeled eval set
    (EVAL_QUERY_SET_PATH) if present, else falls back to a small built-in
    sample so this always runs even before you've written real queries.

    Returns hit rate, avg latency for hits vs. misses, and the resulting
    speedup factor (miss latency / hit latency) — i.e. exactly the
    "benchmarked for latency and hit-rate improvement" ask.
    """
    if queries is None:
        eval_set = load_eval_query_set()
        queries = (
            [item["query"] for item in eval_set]
            if eval_set
            else [
                "gaming laptop under $1500",
                "best laptop with 16gb ram for programming",
                "budget laptop under $600",
                "laptop with nvidia rtx graphics",
                "lightweight laptop for travel",
            ]
        )

    rng = np.random.default_rng(seed)
    paraphrase_prefixes = ["", "looking for a ", "can you suggest a ", "I need a "]

    stream = []
    for q in queries:
        stream.append(q)  # first occurrence -> guaranteed miss
        if rng.random() < repeat_ratio:
            prefix = paraphrase_prefixes[rng.integers(len(paraphrase_prefixes))]
            stream.append((prefix + q).strip())  # near-duplicate -> should hit

    # Fresh cache instance so the benchmark is reproducible and doesn't get
    # polluted by (or pollute) the live global `cache` used by real traffic.
    bench_cache = SemanticCache(similarity_threshold=cache.threshold)

    hit_latencies, miss_latencies = [], []
    hits = 0

    for q in stream:
        query_embedding = embed_model.encode([q])[0].astype("float32")

        t0 = time.perf_counter()
        cached_response, _match_info = bench_cache.lookup(query_embedding)
        if cached_response is not None:
            hit_latencies.append(time.perf_counter() - t0)
            hits += 1
            continue

        result = recommend_verified(q, index_type, preset, k)
        miss_latencies.append(time.perf_counter() - t0)
        bench_cache.store(q, query_embedding, result)

    total = len(stream)
    hit_rate = hits / total if total else 0.0
    avg_hit_latency = float(np.mean(hit_latencies)) if hit_latencies else 0.0
    avg_miss_latency = float(np.mean(miss_latencies)) if miss_latencies else 0.0
    speedup = (avg_miss_latency / avg_hit_latency) if avg_hit_latency > 0 else None

    return {
        "num_queries_in_stream": total,
        "cache_hits": hits,
        "cache_misses": total - hits,
        "hit_rate": round(hit_rate, 4),
        "avg_latency_cache_hit_sec": round(avg_hit_latency, 5),
        "avg_latency_cache_miss_sec": round(avg_miss_latency, 5),
        "speedup_factor": round(speedup, 2) if speedup else None,
        "similarity_threshold": bench_cache.threshold,
    }


def recommend_cached(query: str, index_type, preset, k: int = 5):
    query_embedding = embed_model.encode([query])[0].astype("float32")

    cached_response, match_info = cache.lookup(query_embedding)
    if cached_response is not None:
        result = dict(cached_response)
        result["cache_hit"] = True
        result["matched_query"] = match_info["matched_query"]
        result["similarity"] = round(match_info["similarity"], 4)
        return result

    result = recommend_verified(query, index_type, preset, k)
    result["cache_hit"] = False
    cache.store(query, query_embedding, result)
    return result


def recommend_full(query: str, index_type, preset, k: int = 5, use_cache: bool = True):
    query_embedding = None
    if use_cache:
        query_embedding = embed_model.encode([query])[0].astype("float32")
        cached_response, match_info = cache.lookup(query_embedding)
        if cached_response is not None:
            result = dict(cached_response)
            result["cache_hit"] = True
            result["matched_query"] = match_info["matched_query"]
            result["similarity"] = round(match_info["similarity"], 4)
            return result

    result = recommend_verified(query, index_type, preset, k)
    result["cache_hit"] = False

    top_cpu = (
        result["retrieved_context"][0]["cpu"] if result["retrieved_context"] else None
    )
    result["commonly_paired_specs"] = get_paired_specs(top_cpu) if top_cpu else []

    if use_cache:
        cache.store(query, query_embedding, result)

    return result


def build_full_response(
    query: str, k: int = 5, index_type: str = "ivfpq", preset: str = "medium"
):
    """
    One-stop, everything-about-this-query response. Ties together every
    piece the project spec asks for in a single call, uncached (always
    fresh) so timings and grounding reflect this exact request:

      - retrieved_context: the FAISS + metadata-filtered candidate devices,
        returned alongside the answer as section 3.4 requires ('Retrieved
        context must be displayed alongside the generated answer').
      - recommendation: the LLM's answer, grounded in retrieved_context.
      - grounding_check: the hallucination check from verify_grounding()
        (section 3.4: 'unsupported/hallucinated answers are not
        acceptable') — verified_facts, warnings, and a pass/fail flag.
      - commonly_paired_specs: FP-Growth-style paired-spec insights
        (section 4 stretch goal), computed for *every* unique CPU among
        the retrieved devices (not just the top one), keyed by CPU.
      - timing_sec: retrieval vs. generation latency breakdown, useful
        alongside /benchmark and /benchmark_cache for your report.
    """
    t0 = time.perf_counter()
    retrieved, applied_filters = retrieve(query, k, index_type, preset)
    retrieve_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    answer = generate_recommendation(query, retrieved)
    generation_time = time.perf_counter() - t1

    verified_facts, warnings = verify_grounding(
        query, retrieved, applied_filters, answer
    )

    unique_cpus = [c for c in retrieved["cpu"].dropna().unique().tolist()]
    paired_specs_by_cpu = {cpu: get_paired_specs(cpu, top_n=3) for cpu in unique_cpus}

    return {
        "query": query,
        "index_type": index_type,
        "preset": preset,
        "k": k,
        "filters_applied": applied_filters,
        "retrieved_context": _records_safe(retrieved.drop(columns=["lineage"])),
        "lineage": retrieved["lineage"].tolist(),
        "recommendation": answer,
        "grounding_check": {
            "verified_facts": verified_facts,
            "warnings": warnings,
            "passed": len(warnings) == 0,
        },
        "commonly_paired_specs": paired_specs_by_cpu,
        "timing_sec": {
            "retrieval": round(retrieve_time, 4),
            "generation": round(generation_time, 4),
            "total": round(retrieve_time + generation_time, 4),
        },
    }


# ---------------------------------------------------------------------------
# 3.6 Evaluation: precision@k / recall@k over a hand-labeled query set
# ---------------------------------------------------------------------------
def load_eval_query_set():
    if os.path.exists(EVAL_QUERY_SET_PATH):
        with open(EVAL_QUERY_SET_PATH) as f:
            return json.load(f)
    return []


def precision_recall_at_k(retrieved_titles, relevant_titles, k):
    retrieved_k = retrieved_titles[:k]
    relevant_set = set(relevant_titles)
    hits = sum(1 for t in retrieved_k if t in relevant_set)
    precision = hits / max(1, len(retrieved_k))
    recall = hits / len(relevant_set) if relevant_set else 0.0
    return precision, recall


def evaluate_retrieval(k: int = 5, index_type: str = "ivfpq", preset: str = "medium"):
    """
    Section 3.6: 'A hand-labeled query set of at least 30 queries with
    expected relevant devices. Report precision@k and recall@k.'

    Reads EVAL_QUERY_SET_PATH — a JSON file you maintain with your team's
    real labeled queries in the form:
        [
          {"query": "gaming laptop under $1500", "relevant_titles": ["Acer Nitro 5", "..."]},
          ...
        ]
    (>= 30 entries for the actual submission). This function does NOT
    invent labels for you — that has to reflect your team's judgment of
    what's actually relevant in your dataset.
    """
    query_set = load_eval_query_set()
    if not query_set:
        return {
            "error": f"No labeled query set found at {EVAL_QUERY_SET_PATH}.",
            "expected_format": [
                {"query": "string", "relevant_titles": ["string", "..."]}
            ],
            "note": "Create this file with >= 30 hand-labeled queries before grading (section 3.6).",
        }

    rows = []
    for item in query_set:
        retrieved_df, _ = retrieve(
            item["query"], k=k, index_type=index_type, preset=preset
        )
        retrieved_titles = retrieved_df["title"].tolist()
        p, r = precision_recall_at_k(
            retrieved_titles, item.get("relevant_titles", []), k
        )
        rows.append(
            {
                "query": item["query"],
                "precision_at_k": round(p, 3),
                "recall_at_k": round(r, 3),
            }
        )

    avg_p = float(np.mean([r["precision_at_k"] for r in rows]))
    avg_r = float(np.mean([r["recall_at_k"] for r in rows]))

    return {
        "index_type": index_type,
        "preset": preset,
        "k": k,
        "num_queries": len(rows),
        "avg_precision_at_k": round(avg_p, 4),
        "avg_recall_at_k": round(avg_r, 4),
        "per_query": rows,
    }


# ---------------------------------------------------------------------------
# 3.5 FastAPI service
# ---------------------------------------------------------------------------
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


class IndexType(str, Enum):
    flat = "flat"
    ivfflat = "ivfflat"
    pq = "pq"
    ivfpq = "ivfpq"
    hnsw = "hnsw"


class Query(BaseModel):
    query: str
    k: int = 5
    index_type: IndexType = IndexType.ivfpq
    preset: IndexPreset = IndexPreset.MEDIUM
    use_cache: bool = True


class FullQueryRequest(BaseModel):
    query: str
    k: int = 5
    index_type: IndexType = IndexType.ivfpq
    preset: IndexPreset = IndexPreset.MEDIUM


class BenchmarkRequest(BaseModel):
    sample_size: int = 200
    k: int = 10
    preset: IndexPreset = IndexPreset.MEDIUM
    index_types: Optional[list[IndexType]] = None


class EvaluateRequest(BaseModel):
    k: int = 5
    index_type: IndexType = IndexType.ivfpq
    preset: IndexPreset = IndexPreset.MEDIUM


class CacheBenchmarkRequest(BaseModel):
    queries: Optional[list[str]] = None
    repeat_ratio: float = 0.5
    k: int = 5
    index_type: IndexType = IndexType.ivfpq
    preset: IndexPreset = IndexPreset.MEDIUM


app = FastAPI(
    title="RAG Laptop/Mobile Recommender API",
    description="Query -> embed -> FAISS retrieve (+ metadata filter) -> LLM recommendation, "
    "with grounding checks, index benchmarking, retrieval evaluation, semantic caching, "
    "and FP-Growth-based paired-spec insights.",
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # tighten to your frontend's domain before deploying for real
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def root():
    return {
        "status": "ok",
        "endpoints": {
            "POST /query": "{'query': '...', 'k': 5, 'index_type': 'ivfpq', 'preset': 'medium'} -> everything for one query: context, grounded answer, hallucination check, FP-Growth paired specs, timing (uncached)",
            "POST /recommend": "same as /query but cached: {'query': '...', 'k': 5, 'index_type': 'ivfpq', 'preset': 'medium', 'use_cache': true}",
            "POST /benchmark": "{'sample_size': 200, 'k': 10, 'preset': 'medium'} -> recall@k + latency per index type",
            "POST /evaluate": "{'k': 5, 'index_type': 'ivfpq', 'preset': 'medium'} -> precision@k/recall@k over labeled query set",
            "GET /paired_specs/{cpu}": "commonly paired GPU/RAM/storage for a given CPU (FP-Growth-style)",
            "POST /benchmark_cache": "{'queries': [...] or null, 'repeat_ratio': 0.5} -> semantic cache hit-rate + latency benchmark",
            "GET /health": "liveness + dataset/index status",
        },
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "num_chunks": int(len(df)),
        "num_devices": int(len(device_level)),
        "embedding_dim": int(embeddings.shape[1]),
        "built_indexes": {
            itype: list(presets.keys()) for itype, presets in indexes.items()
        },
        "cache_size": len(cache),
        "has_paired_specs_table": paired_specs_df is not None,
    }


@app.post("/recommend")
def api_recommend(q: Query):
    try:
        return recommend_full(
            query=q.query,
            k=q.k,
            index_type=q.index_type.value,
            preset=q.preset.value,
            use_cache=q.use_cache,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/query")
def api_full_query(req: FullQueryRequest):
    """
    All-in-one endpoint: point your frontend at this one for the main chat
    experience. For a single query, returns in one response:
      - retrieved_context (+ lineage) — shown alongside the answer per 3.4
      - recommendation — the LLM's grounded answer
      - grounding_check — hallucination check (verified_facts / warnings / passed)
      - commonly_paired_specs — FP-Growth-style insights per retrieved CPU
      - timing_sec — retrieval vs. generation latency
    Always runs fresh (no semantic cache), so grounding and timings reflect
    exactly this request. Use /recommend instead if you want cache hits.
    """
    try:
        return build_full_response(
            query=req.query,
            k=req.k,
            index_type=req.index_type.value,
            preset=req.preset.value,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/benchmark")
def api_benchmark(req: BenchmarkRequest):
    """Section 3.3: compare FAISS index types on recall@k and query latency."""
    try:
        index_types = [t.value for t in req.index_types] if req.index_types else None
        return benchmark_indexes(
            sample_size=req.sample_size,
            k=req.k,
            index_types=index_types,
            preset=req.preset.value,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/evaluate")
def api_evaluate(req: EvaluateRequest):
    """Section 3.6: precision@k / recall@k over your hand-labeled query set."""
    return evaluate_retrieval(
        k=req.k, index_type=req.index_type.value, preset=req.preset.value
    )


@app.get("/paired_specs/{cpu}")
def api_paired_specs(cpu: str, top_n: int = 5):
    return {"cpu": cpu, "paired_specs": get_paired_specs(cpu, top_n=top_n)}


@app.post("/benchmark_cache")
def api_benchmark_cache(req: CacheBenchmarkRequest):
    """Section 4 stretch goal: semantic cache hit-rate and latency benchmark."""
    try:
        return benchmark_semantic_cache(
            queries=req.queries,
            repeat_ratio=req.repeat_ratio,
            index_type=req.index_type.value,
            preset=req.preset.value,
            k=req.k,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ---------------------------------------------------------------------------
# Startup: warm ONE default index (fast) instead of all 25 combinations
# (the old eager loop). Everything else is built lazily on first use.
# ---------------------------------------------------------------------------
@app.on_event("startup")
def warm_default_index():
    try:
        print("[startup] warming default index ivfpq/medium ...")
        _ = create_index(preset="medium", index_type="ivfpq")
        _.build(embeddings)
        indexes["ivfpq"]["medium"] = _
        print("[startup] default index ready")
    except Exception as e:
        print(
            f"[startup] warm-up failed (non-fatal, will build lazily on first request): {e}"
        )


# ---------------------------------------------------------------------------
# Entrypoint
# ---------------------------------------------------------------------------
if RUNTIME == "kaggle":
    # Kaggle-notebook deployment: expose the API publicly via ngrok.
    import nest_asyncio
    import uvicorn
    from pyngrok import ngrok
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("NGROK_TOK")

    ngrok.set_auth_token(secret_value_0)
    nest_asyncio.apply()

    # NOTE: the original code called ngrok.disconnect(tunnel.public_url)
    # here, BEFORE `tunnel` was ever created below — that was a guaranteed
    # NameError. A tunnel is opened exactly once now.
    tunnel = ngrok.connect(8000)
    PUBLIC_URL = tunnel.public_url
    print("Public URL:", PUBLIC_URL)

    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True
    ).start()

else:
    # Plain FastAPI service: `python main.py` or `uvicorn main:app --host 0.0.0.0 --port 8000`
    if __name__ == "__main__":
        import uvicorn

        uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", 8000)))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[startup] loaded 4171 chunk rows, 1960 unique devices, embedding dim=384
[startup] no source-link column found (looked for: ['source_url', 'source_link', 'product_url', 'url', 'link']) — 'source_url' will be null for every device. Add one of these column names to your collected dataset to enable device source links.
[startup] no paired-specs table found at /kaggle/input/datasets/mrnotalent/laptop-embedding/paired_specs.parquet — 'commonly paired specs' will fall back to on-the-fly co-occurrence counts. Run the offline Spark FPGrowth job to populate this file for the real stretch-goal credit.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[startup] LLM device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


/tmp/ipykernel_58/348416800.py:1406: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


Public URL: https://unmade-prognosis-savanna.ngrok-free.dev                                         


In [4]:
# ngrok.disconnect(tunnel.public_url)

INFO:     Started server process [58]
INFO:     Waiting for application startup.


[startup] warming default index ivfpq/medium ...


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[startup] default index ready
INFO:     103.180.245.252:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.180.245.252:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.180.245.252:0 - "OPTIONS /recommend HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

INFO:     103.72.212.28:0 - "OPTIONS /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /evaluate HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "POST /evaluate HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "POST /recommend HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /paired_specs/Intel%20Core%20i7-13700HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/Intel%20Core%20i7-13700HX HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     103.72.212.26:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.72.212.26:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD%20Ryzen%207%207745HX HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /paired_specs/AMD HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/AMD HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /paired_specs/Apple%20M3 HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/Apple%20M3 HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "GET /paired_specs/Apple%20M3 HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "OPTIONS /evaluate HTTP/1.1" 200 OK
INFO:     103.72.212.28:0 - "POST /evaluate HTTP/1.1" 200 OK
